In [8]:
!pip install rdkit scikit-learn

In [19]:
!pip install pubchempy

In [20]:
import pubchempy as pcp
import pandas as pd
import time

from rdkit import Chem
from rdkit.Chem import Descriptors

print("All imports successful!")

All imports successful!


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [22]:
molecules = [

"Aspirin","Ibuprofen","Paracetamol","Caffeine","Nicotine",
"Metformin","Atorvastatin","Warfarin","Diazepam","Morphine",
"Codeine","Amoxicillin","Ampicillin","Penicillin G","Omeprazole",
"Captopril","Naproxen","Ketoprofen","Lidocaine","Quinine",

"Benzene","Toluene","Phenol","Acetone","Ethanol",
"Methanol","Formaldehyde","Acetaldehyde","Acetic acid","Citric acid",
"Aniline","Naphthalene","Urea","Glucose","Fructose",
"Sucrose","Lactose","Glycerol","Propanol","Butanol",

"Glycine","Alanine","Valine","Leucine","Serine",
"Tyrosine","Histidine","Lysine","Arginine","Methionine",

"Vitamin C","Vitamin A","Vitamin B1","Vitamin B6","Vitamin E",

"Menthol","Vanillin","Limonene","Camphor","Eugenol",
"Curcumin","Capsaicin","Cinnamaldehyde","Resveratrol","Quercetin",

"Theobromine","Theophylline","Cocaine","Atropine","Scopolamine",
"Papaverine","Berberine","Strychnine",

"Cholesterol","Testosterone","Progesterone","Estradiol","Cortisone",
"Oleic acid","Palmitic acid","Stearic acid","Linoleic acid","Triolein",

"Ethylene glycol","Propylene glycol","Bisphenol A","Styrene","Acrylamide",
"Phenanthrene","Anthracene","Pyrene","Benzoic acid","Salicylic acid",

"Atrazine","Glyphosate","Malathion","Carbaryl","Paraquat",

"Acetamide","Benzaldehyde","Acetonitrile","Pyridine","Piperidine",
"Imidazole","Furan","Thiophene","Cyclohexane","Hexane"
]

print("Number of molecules =", len(molecules))

Number of molecules = 108


In [23]:
data = []
for name in molecules:

    try:
      compound = pcp.get_compounds(name, "name")[0]
      smiles = compound.smiles
      formula = compound.molecular_formula
      mol = Chem.MolFromSmiles(smiles)

      mw = Descriptors.MolWt(mol)
      tpsa = Descriptors.TPSA(mol)
      hbd = Descriptors.NumHDonors(mol)
      hba = Descriptors.NumHAcceptors(mol)
      logp = Descriptors.MolLogP(mol)

      data.append([name,formula,mw,tpsa,hbd,hba,logp])

      print("Added:", name)
      time.sleep(0.2)

    except Exception:
      print("Failed:", name)


Added: Aspirin
Added: Ibuprofen
Added: Paracetamol
Added: Caffeine
Added: Nicotine
Added: Metformin
Added: Atorvastatin
Added: Warfarin
Added: Diazepam
Added: Morphine
Added: Codeine
Added: Amoxicillin
Added: Ampicillin
Added: Penicillin G
Added: Omeprazole
Added: Captopril
Added: Naproxen
Added: Ketoprofen
Added: Lidocaine
Added: Quinine
Added: Benzene
Added: Toluene
Added: Phenol
Added: Acetone
Added: Ethanol
Added: Methanol
Added: Formaldehyde
Added: Acetaldehyde
Added: Acetic acid
Added: Citric acid
Added: Aniline
Added: Naphthalene
Added: Urea
Added: Glucose
Added: Fructose
Added: Sucrose
Added: Lactose
Added: Glycerol
Added: Propanol
Added: Butanol
Added: Glycine
Added: Alanine
Added: Valine
Added: Leucine
Added: Serine
Added: Tyrosine
Added: Histidine
Added: Lysine
Added: Arginine
Added: Methionine
Added: Vitamin C
Added: Vitamin A
Added: Vitamin B1
Added: Vitamin B6
Added: Vitamin E
Added: Menthol
Added: Vanillin
Added: Limonene
Added: Camphor
Added: Eugenol
Added: Curcumin
Add

In [25]:
df = pd.DataFrame(data,
                  columns=["Name","Formula","MW","TPSA","HBD","HBA","LogP"])

print(df.shape)
print(df)

(108, 7)
            Name    Formula       MW   TPSA  HBD  HBA    LogP
0        Aspirin     C9H8O4  180.159  63.60    1    3  1.3101
1      Ibuprofen   C13H18O2  206.285  37.30    1    1  3.0732
2    Paracetamol    C8H9NO2  151.165  49.33    2    2  1.3506
3       Caffeine  C8H10N4O2  194.194  61.82    0    3 -1.0293
4       Nicotine   C10H14N2  162.236  16.13    0    2  1.8483
..           ...        ...      ...    ...  ...  ...     ...
103    Imidazole     C3H4N2   68.079  28.68    1    1  0.4097
104        Furan      C4H4O   68.075  13.14    0    1  1.2796
105    Thiophene      C4H4S   84.143   0.00    0    1  1.7481
106  Cyclohexane      C6H12   84.162   0.00    0    0  2.3406
107       Hexane      C6H14   86.178   0.00    0    0  2.5866

[108 rows x 7 columns]


In [26]:
df.to_csv("druglike_dataset.csv", index=False)
print("Dataset saved successfully")

Dataset saved successfully


In [27]:
def drug_like(row):

    if (
        row["MW"] <= 500 and
        row["LogP"] <= 5 and
        row["HBD"] <= 5 and
        row["HBA"] <= 10
    ):
        return 1

    else:
        return 0


df["DrugLike"] = df.apply(
    drug_like,
    axis=1
)

print(df[["Name", "DrugLike"]].head())

          Name  DrugLike
0      Aspirin         1
1    Ibuprofen         1
2  Paracetamol         1
3     Caffeine         1
4     Nicotine         1


In [28]:
def drug_like(row):

    if (
        row["MW"] <= 500 and
        row["LogP"] <= 5 and
        row["HBD"] <= 5 and
        row["HBA"] <= 10
    ):
        return 1

    else:
        return 0


df["DrugLike"] = df.apply(
    drug_like,
    axis=1
)

print(df[["Name", "DrugLike"]])

            Name  DrugLike
0        Aspirin         1
1      Ibuprofen         1
2    Paracetamol         1
3       Caffeine         1
4       Nicotine         1
..           ...       ...
103    Imidazole         1
104        Furan         1
105    Thiophene         1
106  Cyclohexane         1
107       Hexane         1

[108 rows x 2 columns]


In [29]:
print(df["DrugLike"].value_counts())

DrugLike
1    97
0    11
Name: count, dtype: int64


In [30]:
X = df[["MW","TPSA","HBD","HBA","LogP"]]
y = df["DrugLike"]

print(X.shape)
print(y.shape)

(108, 5)
(108,)


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(86, 5)
(22, 5)


In [32]:
model = RandomForestClassifier(random_state=42)

model.fit(X_train,y_train)

print("Model trained successfully!")

Model trained successfully!


In [33]:
predictions = model.predict(X_test)

print(predictions)

[1 1 1 1 1 1 1 1 1 1 0 0 1 0 1 1 1 1 1 1 1 0]


In [34]:
accuracy = accuracy_score(y_test,predictions)

print("Accuracy =", accuracy)

Accuracy = 1.0


In [35]:
print(df["DrugLike"].value_counts())

DrugLike
1    97
0    11
Name: count, dtype: int64


In [36]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [37]:
y_pred = model.predict(X_test)

In [38]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 4  0]
 [ 0 18]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00        18

    accuracy                           1.00        22
   macro avg       1.00      1.00      1.00        22
weighted avg       1.00      1.00      1.00        22

